# 01 · Data Cleaning and Preparation

**Hong Kong Cross-Border Passenger Traffic Analysis**  
Vila Chung · HKU BASc Social Data Science · 2025

---

## CRISP-DM Phase: Data Preparation

This notebook covers the **Data Preparation** phase of the CRISP-DM framework:

1. Load and inspect the raw Immigration Department dataset
2. Handle missing values and data type issues
3. Aggregate control-point-level records into daily totals
4. Engineer temporal, holiday, and festival features
5. Filter to the post-reopening period (2023–2025) — excluding the COVID structural break
6. Discretise the target variable for classification tasks
7. Export a clean, analysis-ready dataset

**Input:** `statistics_on_daily_passenger_traffic.csv` (56,424 rows)  
**Output:** `daily_traffic_processed.csv`

### Unified Holiday Column Schema (created in this notebook)

| Column | Definition |
|--------|------------|
| `Is_HK_Holiday` | 1 if gazetted Hong Kong public holiday |
| `Is_ML_Holiday` | 1 if Mainland China public holiday |
| `Is_Both_Holiday` | 1 if **both** HK and Mainland holiday on same day |
| `Is_Any_Holiday` | 1 if **either** HK or Mainland holiday |
| `Is_Holiday` | Alias for `Is_Any_Holiday` (for concise downstream use) |

> All downstream notebooks (02–05) and app.py read these columns directly from the CSV.  
> No holiday reconstruction is needed anywhere else.

---
## 0. Environment Setup

In [22]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Libraries loaded successfully.')
print(f'pandas version: {pd.__version__}')
print(f'numpy version:  {np.__version__}')

Libraries loaded successfully.
pandas version: 3.0.1
numpy version:  2.4.3


---
## 1. Load Raw Data

The dataset is sourced from the **Hong Kong Immigration Department** via
[data.gov.hk](https://data.gov.hk/en-data/dataset/hk-immd-set4-statistics-daily-passenger-traffic).
It contains daily passenger traffic statistics across all border control points,
broken down by:

- **Direction:** Arrival / Departure
- **Traveller type:** Hong Kong Residents, Mainland Visitors, Other Visitors
- **Control point:** 17 boundary checkpoints (Airport, Lo Wu, Shenzhen Bay, etc.)

In [23]:
# Load raw CSV
raw_path = 'statistics_on_daily_passenger_traffic.csv'
df_raw = pd.read_csv(raw_path)

print(f'Dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Memory usage: {df_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB')
df_raw.head(10)

Dataset loaded: 56,424 rows x 8 columns
Memory usage: 5.53 MB


,Date,Control Point,Arrival / Departure,Hong Kong Residents,Mainland Visitors,Other Visitors,Total,Unnamed: 7
0,01-01-2021,Airport,Arrival,341,0,9,350,NaN
1,01-01-2021,Airport,Departure,803,17,28,848,NaN
2,01-01-2021,Express Rail Link West Kowloon,Arrival,0,0,0,0,NaN
3,01-01-2021,Express Rail Link West Kowloon,Departure,0,0,0,0,NaN
4,01-01-2021,Hung Hom,Arrival,0,0,0,0,NaN
5,01-01-2021,Hung Hom,Departure,0,0,0,0,NaN
6,01-01-2021,Lo Wu,Arrival,0,0,0,0,NaN
7,01-01-2021,Lo Wu,Departure,0,0,0,0,NaN
8,01-01-2021,Lok Ma Chau Spur Line,Arrival,0,0,0,0,NaN
9,01-01-2021,Lok Ma Chau Spur Line,Departure,0,0,0,0,NaN


---
## 2. Basic Inspection

In [24]:
# 2.1 Shape and column overview
print('=' * 65)
print('DATASET OVERVIEW')
print('=' * 65)
print(f'\nShape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'\nColumns:')
for i, col in enumerate(df_raw.columns, 1):
    print(f'  {i:2d}. {col}')
print(f'\nData types:')
print(df_raw.dtypes)

DATASET OVERVIEW

Shape: 56,424 rows x 8 columns

Columns:
   1. Date
   2. Control Point
   3. Arrival / Departure
   4. Hong Kong Residents
   5. Mainland Visitors
   6. Other Visitors
   7. Total
   8. Unnamed: 7

Data types:
Date                       str
Control Point              str
Arrival / Departure        str
Hong Kong Residents      int64
Mainland Visitors        int64
Other Visitors           int64
Total                    int64
Unnamed: 7             float64
dtype: object


In [25]:
# 2.2 Missing values
print('=' * 65)
print('MISSING VALUES')
print('=' * 65)
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df)
print(f'\nTotal missing cells: {df_raw.isnull().sum().sum():,}')

MISSING VALUES
                     Missing Count  Missing %
Date                             0       0.00
Control Point                    0       0.00
Arrival / Departure              0       0.00
Hong Kong Residents              0       0.00
Mainland Visitors                0       0.00
Other Visitors                   0       0.00
Total                            0       0.00
Unnamed: 7                   56424     100.00

Total missing cells: 56,424


In [26]:
# 2.3 Descriptive statistics
print('=' * 65)
print('DESCRIPTIVE STATISTICS (numeric columns)')
print('=' * 65)
df_raw.describe()

DESCRIPTIVE STATISTICS (numeric columns)


,Hong Kong Residents,Mainland Visitors,Other Visitors,Total,Unnamed: 7
count,"56,424.00","56,424.00","56,424.00","56,424.00",0.00
mean,"11,326.93","3,811.72","1,146.32","16,284.97",NaN
std,"20,026.79","6,723.72","3,478.00","26,833.22",NaN
min,0.00,0.00,0.00,0.00,NaN
25%,0.00,0.00,0.00,0.00,NaN
50%,376.00,23.00,16.00,697.00,NaN
75%,"15,663.00","5,534.50",570.00,"27,036.00",NaN
max,"155,071.00","65,939.00","36,173.00","168,435.00",NaN


In [27]:
# 2.4 Unique values per column
print('=' * 65)
print('UNIQUE VALUES')
print('=' * 65)
for col in df_raw.columns:
    n_unique = df_raw[col].nunique()
    print(f'  {col:30s} : {n_unique:>6,} unique values')

print(f'\nControl Points ({df_raw["Control Point"].nunique()}):')
for cp in sorted(df_raw['Control Point'].unique()):
    print(f'  - {cp}')

print(f'\nArrival / Departure values:')
print(f'  {df_raw["Arrival / Departure"].unique()}')

UNIQUE VALUES
  Date                           :  1,893 unique values
  Control Point                  :     17 unique values
  Arrival / Departure            :      2 unique values
  Hong Kong Residents            : 20,204 unique values
  Mainland Visitors              : 14,107 unique values
  Other Visitors                 :  6,736 unique values
  Total                          : 21,758 unique values
  Unnamed: 7                     :      0 unique values

Control Points (17):
  - Airport
  - China Ferry Terminal
  - Express Rail Link West Kowloon
  - Harbour Control
  - Heung Yuen Wai
  - Hong Kong-Zhuhai-Macao Bridge
  - Hung Hom
  - Kai Tak Cruise Terminal
  - Lo Wu
  - Lok Ma Chau
  - Lok Ma Chau Spur Line
  - Macao Ferry Terminal
  - Macau Ferry Terminal
  - Man Kam To
  - Sha Tau Kok
  - Shenzhen Bay
  - Tuen Mun Ferry Terminal

Arrival / Departure values:
  <ArrowStringArray>
['Arrival', 'Departure']
Length: 2, dtype: str


---
## 3. Data Cleaning

In [28]:
# 3.1 Drop the empty artefact column
print('Columns before cleaning:', list(df_raw.columns))
df = df_raw.drop(columns=['Unnamed: 7'], errors='ignore').copy()
print(f'\nDropped "Unnamed: 7" -> {df.shape[1]} columns remain.')

# 3.2 Parse Date column to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
print(f'\nDate range : {df["Date"].min().date()}  to  {df["Date"].max().date()}')
print(f'Date dtype : {df["Date"].dtype}')

# Verify no remaining missing values in core columns
core_cols = ['Date', 'Control Point', 'Arrival / Departure',
             'Hong Kong Residents', 'Mainland Visitors',
             'Other Visitors', 'Total']
remaining_missing = df[core_cols].isnull().sum().sum()
print(f'\nRemaining missing values in core columns: {remaining_missing}')

print('\nCleaned DataFrame sample:')
df.head()

Columns before cleaning: ['Date', 'Control Point', 'Arrival / Departure', 'Hong Kong Residents', 'Mainland Visitors', 'Other Visitors', 'Total', 'Unnamed: 7']

Dropped "Unnamed: 7" -> 7 columns remain.

Date range : 2021-01-01  to  2026-03-08
Date dtype : datetime64[us]

Remaining missing values in core columns: 0

Cleaned DataFrame sample:


,Date,Control Point,Arrival / Departure,Hong Kong Residents,Mainland Visitors,Other Visitors,Total
0,2021-01-01,Airport,Arrival,341,0,9,350
1,2021-01-01,Airport,Departure,803,17,28,848
2,2021-01-01,Express Rail Link West Kowloon,Arrival,0,0,0,0
3,2021-01-01,Express Rail Link West Kowloon,Departure,0,0,0,0
4,2021-01-01,Hung Hom,Arrival,0,0,0,0


---
## 4. Daily Aggregation

In [29]:
# Passenger columns to aggregate
passenger_cols = ['Hong Kong Residents', 'Mainland Visitors',
                  'Other Visitors', 'Total']

# Aggregate to daily totals
df_daily = df.groupby('Date')[passenger_cols].sum().reset_index()
df_daily = df_daily.sort_values('Date').reset_index(drop=True)

print(f'Aggregated: {df.shape[0]:,} rows  ->  {df_daily.shape[0]:,} daily records')
print(f'Date range: {df_daily["Date"].min().date()}  to  {df_daily["Date"].max().date()}')
print(f'\nDaily total statistics:')
df_daily[passenger_cols].describe().round(0)

Aggregated: 56,424 rows  ->  1,893 daily records
Date range: 2021-01-01  to  2026-03-08

Daily total statistics:


,Hong Kong Residents,Mainland Visitors,Other Visitors,Total
count,"1,893.00","1,893.00","1,893.00","1,893.00"
mean,"337,618.00","113,615.00","34,168.00","485,401.00"
std,"293,285.00","105,775.00","29,573.00","420,115.00"
min,"1,199.00",41.00,22.00,"1,276.00"
25%,"7,098.00",455.00,138.00,"7,997.00"
50%,"410,237.00","132,482.00","40,361.00","608,827.00"
75%,"573,573.00","188,719.00","59,652.00","831,665.00"
max,"1,022,762.00","519,095.00","109,487.00","1,342,500.00"


---
## 5. Feature Engineering

### 5.1 Temporal Features

In [30]:
# 5.1 Temporal features
df_daily['Year']       = df_daily['Date'].dt.year
df_daily['Month']      = df_daily['Date'].dt.month
df_daily['DayOfWeek']  = df_daily['Date'].dt.dayofweek   # 0=Mon, 6=Sun
df_daily['DayName']    = df_daily['Date'].dt.day_name()
df_daily['Quarter']    = df_daily['Date'].dt.quarter
df_daily['Is_Weekend'] = (df_daily['DayOfWeek'] >= 5).astype(int)

print('Temporal features added:')
print(f'  Year:       {sorted(df_daily["Year"].unique())}')
print(f'  Month:      1 - 12')
print(f'  DayOfWeek:  0 (Mon) - 6 (Sun)')
print(f'  Quarter:    1 - 4')
print(f'  Is_Weekend: {df_daily["Is_Weekend"].value_counts().to_dict()}')

Temporal features added:
  Year:       [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
  Month:      1 - 12
  DayOfWeek:  0 (Mon) - 6 (Sun)
  Quarter:    1 - 4
  Is_Weekend: {0: 1351, 1: 542}


### 5.2 Hong Kong Public Holiday Flag (`Is_HK_Holiday`)

All 93 gazetted Hong Kong public holidays from 2021 to 2025,
sourced from [GovHK General Holidays](https://www.gov.hk/en/about/abouthk/holiday/).

> This column was previously named `Is_Holiday`. It is now renamed `Is_HK_Holiday`  
> for clarity. The alias `Is_Holiday = Is_Any_Holiday` is created in Section 5.5.

In [31]:
# 5.2 Hong Kong Public Holidays — gazetted list 2021-2025
# Source: https://www.gov.hk/en/about/abouthk/holiday/

hk_holidays = [
    # ── 2021 (19 days) ──────────────────────────────────────
    '2021-01-01',  # New Year's Day
    '2021-02-12',  # Lunar New Year Day 1
    '2021-02-13',  # Lunar New Year Day 2
    '2021-02-14',  # Lunar New Year Day 3 (Sunday)
    '2021-02-15',  # Lunar New Year Day 3 replacement (Monday)
    '2021-04-02',  # Good Friday
    '2021-04-03',  # Day after Good Friday
    '2021-04-04',  # Ching Ming Festival (Sunday)
    '2021-04-05',  # Easter Monday / Ching Ming replacement
    '2021-05-01',  # Labour Day
    '2021-05-19',  # Buddha's Birthday
    '2021-06-14',  # Tuen Ng Festival
    '2021-07-01',  # HKSAR Establishment Day
    '2021-09-22',  # Day after Mid-Autumn Festival
    '2021-10-01',  # National Day
    '2021-10-14',  # Chung Yeung Festival
    '2021-12-25',  # Christmas Day
    '2021-12-27',  # 1st weekday after Christmas
    '2021-12-28',  # Additional replacement

    # ── 2022 (20 days) ──────────────────────────────────────
    '2022-01-01',  # New Year's Day
    '2022-02-01',  # Lunar New Year Day 1
    '2022-02-02',  # Lunar New Year Day 2
    '2022-02-03',  # Lunar New Year Day 3
    '2022-04-05',  # Ching Ming Festival
    '2022-04-15',  # Good Friday
    '2022-04-16',  # Day after Good Friday
    '2022-04-18',  # Easter Monday
    '2022-05-01',  # Labour Day (Sunday)
    '2022-05-02',  # Labour Day replacement
    '2022-05-08',  # Buddha's Birthday (Sunday)
    '2022-05-09',  # Buddha's Birthday replacement
    '2022-06-03',  # Tuen Ng Festival
    '2022-07-01',  # HKSAR Establishment Day
    '2022-09-12',  # Day after Mid-Autumn Festival
    '2022-10-01',  # National Day
    '2022-10-04',  # Chung Yeung Festival
    '2022-12-25',  # Christmas Day (Sunday)
    '2022-12-26',  # Christmas replacement
    '2022-12-27',  # 1st weekday after Christmas

    # ── 2023 (19 days) ──────────────────────────────────────
    '2023-01-01',  # New Year's Day (Sunday)
    '2023-01-02',  # New Year's Day replacement
    '2023-01-22',  # Lunar New Year Day 1 (Sunday)
    '2023-01-23',  # Lunar New Year Day 2
    '2023-01-24',  # Lunar New Year Day 3
    '2023-01-25',  # Lunar New Year Day 1 replacement
    '2023-04-05',  # Ching Ming Festival
    '2023-04-07',  # Good Friday
    '2023-04-08',  # Day after Good Friday
    '2023-04-10',  # Easter Monday
    '2023-05-01',  # Labour Day
    '2023-05-26',  # Buddha's Birthday
    '2023-06-22',  # Tuen Ng Festival
    '2023-07-01',  # HKSAR Establishment Day
    '2023-09-30',  # Day after Mid-Autumn Festival
    '2023-10-01',  # National Day (Sunday)
    '2023-10-02',  # National Day replacement
    '2023-10-23',  # Chung Yeung Festival
    '2023-12-25',  # Christmas Day

    # ── 2024 (18 days) ──────────────────────────────────────
    '2024-01-01',  # New Year's Day
    '2024-02-10',  # Lunar New Year Day 1
    '2024-02-11',  # Lunar New Year Day 2 (Sunday)
    '2024-02-12',  # Lunar New Year Day 3 / Day 2 replacement
    '2024-02-13',  # Lunar New Year Day 2 replacement
    '2024-03-29',  # Good Friday
    '2024-03-30',  # Day after Good Friday
    '2024-04-01',  # Easter Monday
    '2024-04-04',  # Ching Ming Festival
    '2024-05-01',  # Labour Day
    '2024-05-15',  # Buddha's Birthday
    '2024-06-10',  # Tuen Ng Festival
    '2024-07-01',  # HKSAR Establishment Day
    '2024-09-18',  # Day after Mid-Autumn Festival
    '2024-10-01',  # National Day
    '2024-10-11',  # Chung Yeung Festival
    '2024-12-25',  # Christmas Day
    '2024-12-26',  # 1st weekday after Christmas

    # ── 2025 (17 days) ──────────────────────────────────────
    '2025-01-01',  # New Year's Day
    '2025-01-29',  # Lunar New Year Day 1
    '2025-01-30',  # Lunar New Year Day 2
    '2025-01-31',  # Lunar New Year Day 3
    '2025-04-04',  # Ching Ming Festival
    '2025-04-18',  # Good Friday
    '2025-04-19',  # Day after Good Friday
    '2025-04-21',  # Easter Monday
    '2025-05-01',  # Labour Day
    '2025-05-05',  # Buddha's Birthday
    '2025-05-31',  # Tuen Ng Festival
    '2025-07-01',  # HKSAR Establishment Day
    '2025-10-01',  # National Day
    '2025-10-07',  # Day after Mid-Autumn Festival
    '2025-10-29',  # Chung Yeung Festival
    '2025-12-25',  # Christmas Day
    '2025-12-26',  # 1st weekday after Christmas
]

hk_holidays_dt = pd.to_datetime(hk_holidays)

# Create Is_HK_Holiday (renamed from the original Is_Holiday)
df_daily['Is_HK_Holiday'] = df_daily['Date'].isin(hk_holidays_dt).astype(int)

print(f'Total HK gazetted public holidays labelled: {len(hk_holidays_dt)}')
print(f'Is_HK_Holiday days in dataset: {df_daily["Is_HK_Holiday"].sum()}')
print(f'\nDistribution:')
print(df_daily['Is_HK_Holiday'].value_counts().rename({0: 'Non-Holiday', 1: 'HK Holiday'}))

Total HK gazetted public holidays labelled: 93
Is_HK_Holiday days in dataset: 93

Distribution:
Is_HK_Holiday
Non-Holiday    1800
HK Holiday       93
Name: count, dtype: int64


### 5.3 Mainland China Public Holiday Flag (`Is_ML_Holiday`)

Mainland China public holidays for 2021–2025, sourced from
[english.www.gov.cn](http://english.www.gov.cn).

Mainland holidays covered: Spring Festival (CNY), Qingming, Labour Day,
Dragon Boat, National Day Golden Week, and Mid-Autumn Festival.

> **Why this matters:** When HK and Mainland holidays overlap, cross-border demand
> is amplified from both sides simultaneously — this dual-holiday effect is a key
> driver of peak traffic surges.

In [32]:
# 5.3 Mainland China Public Holidays 2021-2025
# Source: http://english.www.gov.cn (State Council holiday announcements)

ml_holidays = [
    # ── 2021 ────────────────────────────────────────────────
    # New Year
    '2021-01-01',
    # Spring Festival (CNY) — Jan 31 to Feb 12 adjusted
    '2021-02-11', '2021-02-12', '2021-02-13', '2021-02-14',
    '2021-02-15', '2021-02-16', '2021-02-17',
    # Qingming
    '2021-04-03', '2021-04-04', '2021-04-05',
    # Labour Day
    '2021-05-01', '2021-05-02', '2021-05-03', '2021-05-04', '2021-05-05',
    # Dragon Boat
    '2021-06-12', '2021-06-13', '2021-06-14',
    # National Day + Mid-Autumn
    '2021-09-19', '2021-09-20', '2021-09-21',
    '2021-10-01', '2021-10-02', '2021-10-03', '2021-10-04',
    '2021-10-05', '2021-10-06', '2021-10-07',

    # ── 2022 ────────────────────────────────────────────────
    # New Year
    '2022-01-01', '2022-01-02', '2022-01-03',
    # Spring Festival
    '2022-01-31', '2022-02-01', '2022-02-02', '2022-02-03',
    '2022-02-04', '2022-02-05', '2022-02-06',
    # Qingming
    '2022-04-03', '2022-04-04', '2022-04-05',
    # Labour Day
    '2022-04-30', '2022-05-01', '2022-05-02', '2022-05-03', '2022-05-04',
    # Dragon Boat
    '2022-06-03', '2022-06-04', '2022-06-05',
    # Mid-Autumn
    '2022-09-10', '2022-09-11', '2022-09-12',
    # National Day
    '2022-10-01', '2022-10-02', '2022-10-03', '2022-10-04',
    '2022-10-05', '2022-10-06', '2022-10-07',

    # ── 2023 ────────────────────────────────────────────────
    # New Year
    '2023-01-01', '2023-01-02',
    # Spring Festival
    '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-01-25', '2023-01-26', '2023-01-27',
    # Qingming
    '2023-04-05',
    # Labour Day
    '2023-04-29', '2023-04-30', '2023-05-01', '2023-05-02', '2023-05-03',
    # Dragon Boat
    '2023-06-22', '2023-06-23', '2023-06-24',
    # Mid-Autumn + National Day
    '2023-09-29', '2023-09-30',
    '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-04',
    '2023-10-05', '2023-10-06',

    # ── 2024 ────────────────────────────────────────────────
    # New Year
    '2024-01-01',
    # Spring Festival
    '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13',
    '2024-02-14', '2024-02-15', '2024-02-16', '2024-02-17',
    # Qingming
    '2024-04-04', '2024-04-05', '2024-04-06',
    # Labour Day
    '2024-05-01', '2024-05-02', '2024-05-03', '2024-05-04', '2024-05-05',
    # Dragon Boat
    '2024-06-08', '2024-06-09', '2024-06-10',
    # Mid-Autumn
    '2024-09-15', '2024-09-16', '2024-09-17',
    # National Day
    '2024-10-01', '2024-10-02', '2024-10-03', '2024-10-04',
    '2024-10-05', '2024-10-06', '2024-10-07',

    # ── 2025 ────────────────────────────────────────────────
    # New Year
    '2025-01-01',
    # Spring Festival
    '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31',
    '2025-02-01', '2025-02-02', '2025-02-03', '2025-02-04',
    # Qingming
    '2025-04-04', '2025-04-05', '2025-04-06',
    # Labour Day
    '2025-05-01', '2025-05-02', '2025-05-03', '2025-05-04', '2025-05-05',
    # Dragon Boat
    '2025-05-31', '2025-06-01', '2025-06-02',
    # National Day + Mid-Autumn
    '2025-10-01', '2025-10-02', '2025-10-03', '2025-10-04',
    '2025-10-05', '2025-10-06', '2025-10-07', '2025-10-08',
]

ml_holidays_dt = pd.to_datetime(ml_holidays)

# Create Is_ML_Holiday
df_daily['Is_ML_Holiday'] = df_daily['Date'].isin(ml_holidays_dt).astype(int)

print(f'Total Mainland holiday days labelled: {len(set(ml_holidays))}')
print(f'Is_ML_Holiday days in dataset: {df_daily["Is_ML_Holiday"].sum()}')
print(f'\nDistribution:')
print(df_daily['Is_ML_Holiday'].value_counts().rename({0: 'Non-Holiday', 1: 'ML Holiday'}))

Total Mainland holiday days labelled: 144
Is_ML_Holiday days in dataset: 144

Distribution:
Is_ML_Holiday
Non-Holiday    1749
ML Holiday      144
Name: count, dtype: int64


### 5.4 Festival Period Flags

| Festival | Period | Rationale |
|----------|--------|-----------|
| **Chinese New Year** | 3–4 day public holiday cluster | Largest annual cross-border movement |
| **Golden Week** | Oct 1–7 (China National Day week) | Peak outbound travel from Mainland |
| **Easter** | Good Friday → Easter Monday (4 days) | Major HK long weekend |

In [33]:
# 5.4a Chinese New Year periods (HK gazetted CNY holidays)
cny_dates = [
    '2021-02-12', '2021-02-13', '2021-02-14', '2021-02-15',
    '2022-02-01', '2022-02-02', '2022-02-03',
    '2023-01-22', '2023-01-23', '2023-01-24', '2023-01-25',
    '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13',
    '2025-01-29', '2025-01-30', '2025-01-31',
]

# 5.4b Golden Week periods (Oct 1-7 each year)
golden_week_dates = []
for year in range(2021, 2026):
    for day in range(1, 8):
        golden_week_dates.append(f'{year}-10-{day:02d}')

# 5.4c Easter periods (Good Friday to Easter Monday)
easter_dates = [
    '2021-04-02', '2021-04-03', '2021-04-04', '2021-04-05',
    '2022-04-15', '2022-04-16', '2022-04-17', '2022-04-18',
    '2023-04-07', '2023-04-08', '2023-04-09', '2023-04-10',
    '2024-03-29', '2024-03-30', '2024-03-31', '2024-04-01',
    '2025-04-18', '2025-04-19', '2025-04-20', '2025-04-21',
]

# Apply festival flags
df_daily['Is_CNY']        = df_daily['Date'].isin(pd.to_datetime(cny_dates)).astype(int)
df_daily['Is_GoldenWeek'] = df_daily['Date'].isin(pd.to_datetime(golden_week_dates)).astype(int)
df_daily['Is_Easter']     = df_daily['Date'].isin(pd.to_datetime(easter_dates)).astype(int)

print('Festival flags added:')
print(f'  Is_CNY        : {df_daily["Is_CNY"].sum()} days')
print(f'  Is_GoldenWeek : {df_daily["Is_GoldenWeek"].sum()} days')
print(f'  Is_Easter     : {df_daily["Is_Easter"].sum()} days')

Festival flags added:
  Is_CNY        : 18 days
  Is_GoldenWeek : 35 days
  Is_Easter     : 20 days


### 5.5 Unified Holiday Columns

This is the **single source of truth** for all holiday flags used across the project.
All downstream notebooks (02–05) and app.py read these columns directly from the CSV.

| Column | Logic | Use case |
|--------|-------|----------|
| `Is_HK_Holiday` | HK gazetted holiday | HK-side demand analysis |
| `Is_ML_Holiday` | Mainland gazetted holiday | ML-side demand analysis |
| `Is_Both_Holiday` | HK AND Mainland holiday | Dual-holiday surge detection |
| `Is_Any_Holiday` | HK OR Mainland holiday | General holiday effect |
| `Is_Holiday` | Alias = `Is_Any_Holiday` | Concise use in models/dashboard |

In [34]:
# 5.5 Unified holiday columns — single source of truth for all notebooks

# Is_Both_Holiday: day is a public holiday in BOTH HK and Mainland
df_daily['Is_Both_Holiday'] = (
    (df_daily['Is_HK_Holiday'] == 1) & (df_daily['Is_ML_Holiday'] == 1)
).astype(int)

# Is_Any_Holiday: day is a public holiday in EITHER HK or Mainland
df_daily['Is_Any_Holiday'] = (
    (df_daily['Is_HK_Holiday'] == 1) | (df_daily['Is_ML_Holiday'] == 1)
).astype(int)

# Is_Holiday: alias for Is_Any_Holiday (for concise downstream use)
df_daily['Is_Holiday'] = df_daily['Is_Any_Holiday']

# ── Summary report ───────────────────────────────────────
print('=' * 65)
print('UNIFIED HOLIDAY COLUMN SUMMARY')
print('=' * 65)
print(f'\n  Is_HK_Holiday    : {df_daily["Is_HK_Holiday"].sum():>4} days  (HK gazetted holidays)')
print(f'  Is_ML_Holiday    : {df_daily["Is_ML_Holiday"].sum():>4} days  (Mainland gazetted holidays)')
print(f'  Is_Both_Holiday  : {df_daily["Is_Both_Holiday"].sum():>4} days  (Both HK & Mainland holidays)')
print(f'  Is_Any_Holiday   : {df_daily["Is_Any_Holiday"].sum():>4} days  (HK or Mainland holiday)')
print(f'  Is_Holiday       : {df_daily["Is_Holiday"].sum():>4} days  (Alias = Is_Any_Holiday)')

# Cross-tabulation: HK vs ML holiday overlap
print(f'\n  Cross-tabulation (full dataset):')
cross_tab = pd.crosstab(
    df_daily['Is_HK_Holiday'],
    df_daily['Is_ML_Holiday'],
    rownames=['Is_HK_Holiday'],
    colnames=['Is_ML_Holiday']
)
print(cross_tab)

print(f'\n✓ All holiday columns created. No reconstruction needed in downstream files.')

UNIFIED HOLIDAY COLUMN SUMMARY

  Is_HK_Holiday    :   93 days  (HK gazetted holidays)
  Is_ML_Holiday    :  144 days  (Mainland gazetted holidays)
  Is_Both_Holiday  :   53 days  (Both HK & Mainland holidays)
  Is_Any_Holiday   :  184 days  (HK or Mainland holiday)
  Is_Holiday       :  184 days  (Alias = Is_Any_Holiday)

  Cross-tabulation (full dataset):
Is_ML_Holiday     0   1
Is_HK_Holiday          
0              1709  91
1                40  53

✓ All holiday columns created. No reconstruction needed in downstream files.


In [35]:
# Verify the full column set
print('Current columns after feature engineering:')
for i, col in enumerate(df_daily.columns, 1):
    print(f'  {i:2d}. {col:25s}  dtype={df_daily[col].dtype}')

print(f'\nShape: {df_daily.shape[0]:,} rows x {df_daily.shape[1]} columns')
df_daily.head()

Current columns after feature engineering:
   1. Date                       dtype=datetime64[us]
   2. Hong Kong Residents        dtype=int64
   3. Mainland Visitors          dtype=int64
   4. Other Visitors             dtype=int64
   5. Total                      dtype=int64
   6. Year                       dtype=int32
   7. Month                      dtype=int32
   8. DayOfWeek                  dtype=int32
   9. DayName                    dtype=str
  10. Quarter                    dtype=int32
  11. Is_Weekend                 dtype=int64
  12. Is_HK_Holiday              dtype=int64
  13. Is_ML_Holiday              dtype=int64
  14. Is_CNY                     dtype=int64
  15. Is_GoldenWeek              dtype=int64
  16. Is_Easter                  dtype=int64
  17. Is_Both_Holiday            dtype=int64
  18. Is_Any_Holiday             dtype=int64
  19. Is_Holiday                 dtype=int64

Shape: 1,893 rows x 19 columns


,Date,Hong Kong Residents,Mainland Visitors,Other Visitors,Total,Year,Month,DayOfWeek,DayName,Quarter,Is_Weekend,Is_HK_Holiday,Is_ML_Holiday,Is_CNY,Is_GoldenWeek,Is_Easter,Is_Both_Holiday,Is_Any_Holiday,Is_Holiday
0,2021-01-01,4178,290,42,4510,2021,1,4,Friday,1,0,1,1,0,0,0,1,1,1
1,2021-01-02,4845,361,47,5253,2021,1,5,Saturday,1,1,0,0,0,0,0,0,0,0
2,2021-01-03,6462,415,99,6976,2021,1,6,Sunday,1,1,0,0,0,0,0,0,0,0
3,2021-01-04,6131,485,63,6679,2021,1,0,Monday,1,0,0,0,0,0,0,0,0,0
4,2021-01-05,3704,222,80,4006,2021,1,1,Tuesday,1,0,0,0,0,0,0,0,0,0


---
## 6. Filter to Post-Reopening Period (2023–2025)

The border **officially reopened on 8 January 2023**. We filter
to the post-reopening period for all downstream modelling.

> **Decision:** Filter to `Date >= 2023-01-08` (1,096 days of post-reopening data)

In [36]:
# Show pre/post comparison to justify the exclusion
print('=' * 65)
print('DAILY TOTAL TRAFFIC BY YEAR (mean)')
print('=' * 65)
yearly_avg = df_daily.groupby('Year')['Total'].agg(['mean', 'median', 'min', 'max'])
yearly_avg.columns = ['Mean', 'Median', 'Min', 'Max']
print(yearly_avg.round(0).to_string())
print('\n>> 2021-2022 averages are dramatically lower due to COVID border closure.')
print('>> This confirms the structural break and justifies excluding this period.')

DAILY TOTAL TRAFFIC BY YEAR (mean)
           Mean     Median     Min      Max
Year                                       
2021   5,209.00   4,392.00    1276    43399
2022  14,417.00  10,832.00    1654    54553
2023 580,160.00 594,780.00   51809  1069947
2024 815,612.00 780,078.00  437181  1212846
2025 916,847.00 888,156.00   28020  1342500
2026 996,692.00 969,275.00  683347  1284659

>> 2021-2022 averages are dramatically lower due to COVID border closure.
>> This confirms the structural break and justifies excluding this period.


In [37]:
# Filter to post-reopening period
reopening_date = pd.Timestamp('2023-01-08')
df_model = df_daily[df_daily['Date'] >= reopening_date].copy()
df_model = df_model.reset_index(drop=True)

print('=' * 65)
print('FILTERED DATASET (Post-Reopening)')
print('=' * 65)
print(f'Period:    {df_model["Date"].min().date()}  to  {df_model["Date"].max().date()}')
print(f'Records:   {len(df_model):,} days')
print(f'Excluded:  {len(df_daily) - len(df_model):,} days (COVID period 2021-2022)')
print(f'\nYear distribution:')
print(df_model['Year'].value_counts().sort_index())

# Holiday counts in filtered period
print(f'\nHoliday counts in post-reopening period:')
for col in ['Is_HK_Holiday', 'Is_ML_Holiday', 'Is_Both_Holiday', 'Is_Any_Holiday', 'Is_Holiday']:
    print(f'  {col:20s}: {df_model[col].sum():>3} days')

FILTERED DATASET (Post-Reopening)
Period:    2023-01-08  to  2026-03-08
Records:   1,156 days
Excluded:  737 days (COVID period 2021-2022)

Year distribution:
Year
2023    358
2024    366
2025    365
2026     67
Name: count, dtype: int64

Holiday counts in post-reopening period:
  Is_HK_Holiday       :  52 days
  Is_ML_Holiday       :  82 days
  Is_Both_Holiday     :  29 days
  Is_Any_Holiday      : 105 days
  Is_Holiday          : 105 days


---
## 7. Target Variable Discretisation

In [38]:
# 7.1 Four-class target (equal-frequency quantile binning)
df_model['Traffic_Level'] = pd.qcut(
    df_model['Total'],
    q=4,
    labels=['Low', 'Medium', 'High', 'VeryHigh']
)

print('Traffic Level distribution (4-class, equal-frequency):')
print(df_model['Traffic_Level'].value_counts().sort_index())

print(f'\nBin edges (quartile boundaries):')
quantiles = df_model['Total'].quantile([0, 0.25, 0.50, 0.75, 1.0])
for q, val in quantiles.items():
    print(f'  Q{int(q*100):3d}%:  {val:>12,.0f}')

Traffic Level distribution (4-class, equal-frequency):
Traffic_Level
Low         289
Medium      289
High        289
VeryHigh    289
Name: count, dtype: int64

Bin edges (quartile boundaries):
  Q  0%:        28,020
  Q 25%:       646,686
  Q 50%:       780,254
  Q 75%:       946,099
  Q100%:     1,342,500


In [39]:
# 7.2 Binary target for Logistic Regression (High / Low at median)
median_total = df_model['Total'].median()
df_model['Traffic_Binary'] = (df_model['Total'] >= median_total).astype(int)

print(f'Binary split threshold (median): {median_total:,.0f}')
print(f'\nBinary distribution:')
print(f'  High (>= median): {df_model["Traffic_Binary"].sum():,}  ({df_model["Traffic_Binary"].mean():.1%})')
print(f'  Low  (<  median): {(1 - df_model["Traffic_Binary"]).sum():,}  ({1 - df_model["Traffic_Binary"].mean():.1%})')

Binary split threshold (median): 780,254

Binary distribution:
  High (>= median): 578  (50.0%)
  Low  (<  median): 578  (50.0%)


---
## 8. Save Processed Dataset

The output CSV contains **all unified holiday columns** as the single source of truth.
Downstream notebooks (02–05) and app.py read these directly — no reconstruction needed.

In [40]:
# Save processed dataset
output_path = 'daily_traffic_processed.csv'
df_model.to_csv(output_path, index=False)

print(f'Processed dataset saved to: {output_path}')
print(f'Shape: {df_model.shape[0]:,} rows x {df_model.shape[1]} columns')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')
print(f'\nColumns in processed dataset:')
for i, col in enumerate(df_model.columns, 1):
    print(f'  {i:2d}. {col:25s}  dtype={df_model[col].dtype}')

# Final holiday column verification
print(f'\n--- Holiday Column Verification ---')
holiday_cols = ['Is_HK_Holiday', 'Is_ML_Holiday', 'Is_Both_Holiday',
                'Is_Any_Holiday', 'Is_Holiday']
for col in holiday_cols:
    assert col in df_model.columns, f'MISSING: {col}'
    print(f'  ✓ {col:20s} : {df_model[col].sum():>3} holiday days in output')

# Verify Is_Holiday == Is_Any_Holiday
assert (df_model['Is_Holiday'] == df_model['Is_Any_Holiday']).all(), \
    'ERROR: Is_Holiday != Is_Any_Holiday'
print(f'\n  ✓ Is_Holiday == Is_Any_Holiday: VERIFIED')
print(f'\n✓ All checks passed. CSV ready for downstream use.')

Processed dataset saved to: daily_traffic_processed.csv
Shape: 1,156 rows x 21 columns
File size: 95.9 KB

Columns in processed dataset:
   1. Date                       dtype=datetime64[us]
   2. Hong Kong Residents        dtype=int64
   3. Mainland Visitors          dtype=int64
   4. Other Visitors             dtype=int64
   5. Total                      dtype=int64
   6. Year                       dtype=int32
   7. Month                      dtype=int32
   8. DayOfWeek                  dtype=int32
   9. DayName                    dtype=str
  10. Quarter                    dtype=int32
  11. Is_Weekend                 dtype=int64
  12. Is_HK_Holiday              dtype=int64
  13. Is_ML_Holiday              dtype=int64
  14. Is_CNY                     dtype=int64
  15. Is_GoldenWeek              dtype=int64
  16. Is_Easter                  dtype=int64
  17. Is_Both_Holiday            dtype=int64
  18. Is_Any_Holiday             dtype=int64
  19. Is_Holiday                 dtype=int64
 

---
## 9. Data Quality Report

In [41]:
print('=' * 65)
print('         DATA QUALITY REPORT')
print('=' * 65)

# Dataset Overview
print(f'\n  Dataset Overview')
print(f'  {"─" * 55}')
print(f'  Source:         HK Immigration Department (data.gov.hk)')
print(f'  Raw records:    {df_raw.shape[0]:,} rows')
print(f'  Processed:      {len(df_model):,} daily records')
print(f'  Period:         {df_model["Date"].min().date()} to {df_model["Date"].max().date()}')
print(f'  Features:       {df_model.shape[1]} columns')

# Passenger Traffic Summary
print(f'\n  Passenger Traffic Summary (Daily Total)')
print(f'  {"─" * 55}')
print(f'  Mean:           {df_model["Total"].mean():>12,.0f}')
print(f'  Median:         {df_model["Total"].median():>12,.0f}')
print(f'  Std Dev:        {df_model["Total"].std():>12,.0f}')
print(f'  Min:            {df_model["Total"].min():>12,.0f}')
print(f'  Max:            {df_model["Total"].max():>12,.0f}')

# Data Quality Checks
n_expected = (df_model['Date'].max() - df_model['Date'].min()).days + 1
n_actual   = len(df_model)
has_gaps   = n_actual != n_expected
has_dupes  = df_model['Date'].duplicated().sum() > 0
has_nulls  = df_model.isnull().sum().sum() > 0
has_neg    = (df_model[passenger_cols].values < 0).any()

print(f'\n  Data Quality Checks')
print(f'  {"─" * 55}')
print(f'  Missing values:       {df_model.isnull().sum().sum():>6}  {"FAIL" if has_nulls else "PASS"}')
print(f'  Duplicate dates:      {df_model["Date"].duplicated().sum():>6}  {"FAIL" if has_dupes else "PASS"}')
print(f'  Date continuity:      {n_actual}/{n_expected} days  {"GAPS" if has_gaps else "PASS"}')
print(f'  Negative values:      {"YES" if has_neg else "None":>6}  {"FAIL" if has_neg else "PASS"}')

# Feature Summary
print(f'\n  Feature Summary')
print(f'  {"─" * 55}')
print(f'  Temporal:    Year, Month, DayOfWeek, DayName, Quarter, Is_Weekend')
print(f'  HK Holidays: Is_HK_Holiday ({df_model["Is_HK_Holiday"].sum()} days)')
print(f'  ML Holidays: Is_ML_Holiday ({df_model["Is_ML_Holiday"].sum()} days)')
print(f'  Combined:    Is_Both_Holiday ({df_model["Is_Both_Holiday"].sum()} days), '
      f'Is_Any_Holiday ({df_model["Is_Any_Holiday"].sum()} days)')
print(f'  Alias:       Is_Holiday = Is_Any_Holiday ({df_model["Is_Holiday"].sum()} days)')
print(f'  Festivals:   Is_CNY ({df_model["Is_CNY"].sum()}), '
      f'Is_GoldenWeek ({df_model["Is_GoldenWeek"].sum()}), '
      f'Is_Easter ({df_model["Is_Easter"].sum()})')
print(f'  Target:      Traffic_Level (4-class), Traffic_Binary (2-class)')

print(f'\n{"=" * 65}')
print(f'  Data preparation complete. Ready for EDA and modelling.')
print(f'{"=" * 65}')

         DATA QUALITY REPORT

  Dataset Overview
  ───────────────────────────────────────────────────────
  Source:         HK Immigration Department (data.gov.hk)
  Raw records:    56,424 rows
  Processed:      1,156 daily records
  Period:         2023-01-08 to 2026-03-08
  Features:       21 columns

  Passenger Traffic Summary (Daily Total)
  ───────────────────────────────────────────────────────
  Mean:                788,323
  Median:              780,254
  Std Dev:             230,527
  Min:                  28,020
  Max:               1,342,500

  Data Quality Checks
  ───────────────────────────────────────────────────────
  Missing values:            0  PASS
  Duplicate dates:           0  PASS
  Date continuity:      1156/1156 days  PASS
  Negative values:        None  PASS

  Feature Summary
  ───────────────────────────────────────────────────────
  Temporal:    Year, Month, DayOfWeek, DayName, Quarter, Is_Weekend
  HK Holidays: Is_HK_Holiday (52 days)
  ML Holidays: Is_

In [42]:
# Final preview of the processed dataset
print('Processed dataset preview (first 10 rows):')
df_model.head(10)

Processed dataset preview (first 10 rows):


,Date,Hong Kong Residents,Mainland Visitors,Other Visitors,Total,Year,Month,DayOfWeek,DayName,Quarter,Is_Weekend,Is_HK_Holiday,Is_ML_Holiday,Is_CNY,Is_GoldenWeek,Is_Easter,Is_Both_Holiday,Is_Any_Holiday,Is_Holiday,Traffic_Level,Traffic_Binary
0,2023-01-08,82047,13374,13086,108507,2023,1,6,Sunday,1,1,0,0,0,0,0,0,0,0,Low,0
1,2023-01-09,63805,10494,10833,85132,2023,1,0,Monday,1,0,0,0,0,0,0,0,0,0,Low,0
2,2023-01-10,67398,11132,11543,90073,2023,1,1,Tuesday,1,0,0,0,0,0,0,0,0,0,Low,0
3,2023-01-11,67252,12567,12526,92345,2023,1,2,Wednesday,1,0,0,0,0,0,0,0,0,0,Low,0
4,2023-01-12,72996,14736,13225,100957,2023,1,3,Thursday,1,0,0,0,0,0,0,0,0,0,Low,0
5,2023-01-13,83542,16121,14395,114058,2023,1,4,Friday,1,0,0,0,0,0,0,0,0,0,Low,0
6,2023-01-14,91861,17480,17282,126623,2023,1,5,Saturday,1,1,0,0,0,0,0,0,0,0,Low,0
7,2023-01-15,101274,19335,17976,138585,2023,1,6,Sunday,1,1,0,0,0,0,0,0,0,0,Low,0
8,2023-01-16,88040,17736,13778,119554,2023,1,0,Monday,1,0,0,0,0,0,0,0,0,0,Low,0
9,2023-01-17,96328,19140,13108,128576,2023,1,1,Tuesday,1,0,0,0,0,0,0,0,0,0,Low,0


---

**Next:** [02_EDA_and_Statistics.ipynb](02_EDA_and_Statistics.ipynb) →  
Exploratory Data Analysis, statistical testing, and seasonal decomposition on the cleaned dataset.

---

### Downstream Usage Reference

All notebooks and app.py should use the following columns directly from `daily_traffic_processed.csv`:

```python
# Standard load pattern for all downstream files
df = pd.read_csv('daily_traffic_processed.csv', parse_dates=['Date'])

# Available holiday columns (no reconstruction needed)
# df['Is_HK_Holiday']   — HK gazetted holidays only
# df['Is_ML_Holiday']   — Mainland gazetted holidays only
# df['Is_Both_Holiday'] — Both HK and Mainland holiday on same day
# df['Is_Any_Holiday']  — Either HK or Mainland holiday
# df['Is_Holiday']      — Alias for Is_Any_Holiday (recommended for models)
```